## Topic 9: PII and Access-Scoping

### Concept, Intuition, Why It Exists

- `fd_master_database.csv` holds real-shaped customer records: `Customer_Name`, `Mobile_Number`, `Account_Number`, `FD_Amount_INR`, `Status`, `Branch`. When the CSV loader from the ingestion chapter reads this file, every row becomes a Document whose `page_content` is the entire record serialised as `key: value` text. When that Document gets embedded and upserted into Qdrant, the customer's name, phone number, and account number are now sitting inside a searchable vector store — not locked behind a SQL database with row-level access control, but in a collection that any code with a Qdrant client connection can search.
- **The problem**: vector search doesn't respect the identity of the person who submitted the query. A semantic search for "Shobha Chopra FD status" returns Shobha Chopra's record — correct. But a semantic search for "customer with large FD in Pune" can return records for several different customers whose records happen to be semantically similar to that query — records belonging to people who never gave permission for their data to be surfaced to whoever is running this query. The vector store doesn't know who is asking. SQL databases have row-level security, RBAC, audit logs. A naive Qdrant collection has none of these unless you build them explicitly.
- **Metadata filtering is not the same as access control** but it is the mechanism we have in Qdrant that can be used to enforce it — filtering by a `user_id` or `account_owner` payload field so a search only ever returns chunks belonging to the authenticated caller. This topic covers what that looks like in practice, what it doesn't cover, and what a production-grade PII story would need beyond what filtering alone provides.

### The Real Risk In This Project

- The ingestion chapter's CSV loader (Topic 4 of the ingestion sub-chapter) already flagged this: "PII in `page_content` means the vector store inherits the source DB's security requirements." This topic makes that concrete.
- Three failure modes specific to embedding PII into a vector store:
- **Cross-customer leakage**: a query about one customer's FD can semantically match another customer's record if the records are similar enough — same branch, similar amount, similar tenure — and return that other record to whoever is running the query.
- **Unintended surfacing via general queries**: a general query like "what is the current FD interest rate in Pune" intended to search policy documents can, if the customer records are in the same collection as policy documents, semantically match rows from `fd_master_database.csv` and return real customer data as "relevant context" alongside the policy answer.
- **Prompt injection via retrieved context**: a retrieved customer record containing adversarially crafted text in a name or address field gets included in the LLM's context window as "relevant information" and influences the answer in unintended ways.

### How It's Implemented In This Project

- The fix is two-layered:
- **Separation**: customer records (`fd_master_database.csv`) go into a dedicated collection, separate from policy/FAQ/SOP documents. A search against the policy collection can never return customer records, by architecture, not just by convention.
- **Payload filtering for caller scoping**: within the customer records collection, every search is filtered by the authenticated caller's identity — if the call is acting on behalf of FD `BJ2019FD7717`, only the point with `fd_no="BJ2019FD7717"` in its payload is a valid search result. The filtering is done at the Qdrant level during HNSW traversal, not in Python afterward.
- The exact-lookup pattern (`get_fd_record()` from `fd_database.py`) already exists and is the right tool for exact FD_No lookups. The vector search of customer records is only needed for semantic queries across records — and those queries must always be scoped by caller identity.

### Real-World Issues, Edge Cases, Debugging

- **Filtering is not encryption**: payload filtering scopes which results are returned for a given query, but the underlying data in the payload is still stored in plaintext in Qdrant's storage. Anyone with direct access to the Qdrant collection (database-level access) can retrieve any point without going through the application's filtering layer. True data security requires encryption at rest plus access control at the infrastructure level, not just application-level filtering.
- **The "same collection" mistake**: putting customer records and policy documents in the same collection and relying on a `doc_type` filter to keep them separate is operationally fragile — a missing filter in one code path exposes customer data alongside policy content. Separate collections provide architectural separation that a filter dependency doesn't.
- **Right-to-be-forgotten**: GDPR and equivalent regulations require that a customer's data be deletable on request. Deleting a row from `fd_master_database.csv` does not automatically delete the corresponding vector from Qdrant. The application layer must explicitly handle this — delete the source row, then call `client.delete(collection_name, points_selector)` to remove the corresponding point from Qdrant using the same deterministic ID used at upsert time.
- **Embedding PII creates a secondary copy**: the moment a customer record is embedded, a copy of that customer's data exists in two places — the source CSV and the vector store. Any data retention or deletion policy must cover both. Most teams forget the vector store.

### Design Decisions & Trade-offs

- Separate collection for customer records vs. one collection with a `doc_type` filter: separate collection wins for PII specifically, because it provides architectural guarantees rather than filter-dependency guarantees. A bug that causes a missing filter in a policy search will never expose customer records if they physically cannot be in the same collection.
- Exact-match lookup vs. semantic search for customer records: `get_fd_record()` (exact FD_No lookup via the CSV/database) is always the right choice for "look up this specific customer's record." Semantic search of customer records is only appropriate for administrative queries like "find all customers with FDs over Rs 3 lakh in Pune" — and those queries carry stricter access control requirements than customer-facing queries.
- Storing customer records in the vector store at all vs. keeping them only in the structured database: the vector store is only justified for customer records if there is a genuine semantic search use case (not just exact lookups). If the only access pattern is exact FD_No lookup, the records belong in the structured database only and the CSV loader for `fd_master_database.csv` should not upsert into the vector store at all.

### Alternatives & When To Use Each

- Separate Qdrant collection per sensitivity tier — the recommended approach for this project: public knowledge (FAQs, SOPs, policy docs) in one collection, customer PII in a separate collection with caller-scoped filtering.
- Field-level redaction before embedding — embed a version of the customer record with PII fields removed or hashed, keeping only the fields actually needed for semantic search (branch, tenure category, status). The original PII stays only in the structured database.
- Don't embed customer records at all — if all customer record lookups are exact (by FD_No or mobile number), there is no semantic search use case and no justification for the PII to be in the vector store. Use `get_fd_record()` directly.
- Encryption at the Qdrant level — available in Qdrant Cloud's enterprise tier and via volume-level encryption in self-hosted deployments. Necessary for compliance but not a substitute for application-level access scoping.

### Common Mistakes & Production Failures

- Ingesting `fd_master_database.csv` into the same collection as policy documents and assuming a `doc_type` filter is sufficient protection — one code path with a missing filter exposes customer PII to a general policy search.
- Not implementing right-to-be-forgotten for the vector store, covering only the source database, then failing a compliance audit when the PII is found still present in Qdrant.
- Treating "we use filtered search" as equivalent to "we have access control" — filtering is application-layer logic that can have bugs; true access control requires infrastructure-layer enforcement (network access, authentication, encryption at rest).
- Using semantic search for exact-match customer lookup patterns, then not restricting that search to the authenticated caller's own records, allowing cross-customer data leakage.

### Lead-Level Interview Questions

**Q: `fd_master_database.csv` has been ingested into Qdrant. A customer asks "what is my FD status?" and the system runs a semantic search against the full collection. What goes wrong?**
A: Without caller-scoped filtering, the semantic search returns the k most similar customer records to the query — which may include records belonging to different customers with similar FD profiles. The system has potentially surfaced another customer's financial data. The fix is always filtering the customer records collection by the authenticated caller's identity (`fd_no` or a verified `customer_id`) so the search space is scoped to only that caller's records before semantic similarity is evaluated.

**Q: A customer requests deletion of their data under GDPR. You delete their row from `fd_master_database.csv`. Is that sufficient?**
A: No. The moment their record was ingested into Qdrant, a copy of their PII was created in the vector store as a point payload. Deleting the source row leaves their data intact in Qdrant indefinitely. A complete deletion requires: delete the source row, identify the corresponding Qdrant point by its deterministic ID (hash of `FD_No + chunk_index`), and call `client.delete()` on that point. Any data retention policy must explicitly cover the vector store, not just the source database.

**Q: Is payload filtering in Qdrant sufficient as the sole access control mechanism for PII?**
A: No — it is a necessary application-layer safeguard but not sufficient on its own. Payload filtering is application code that can have bugs (a missing filter in one code path). It does not protect data at the storage layer — anyone with direct database access to Qdrant's storage can retrieve any point regardless of what the application's filtering would have done. Real access control requires infrastructure-layer enforcement: network-level access restrictions to the Qdrant instance, authentication, and encryption at rest. Filtering is the application's contract with itself; infrastructure controls are the enforcement mechanism that holds even when application code is wrong.

**Q: Why is a separate collection for customer PII better than a single collection with a doc_type filter?**
A: A filter dependency is application-layer logic — if one code path has a bug and sends a search to the policy FAQ collection without a `doc_type` filter, it can still only return policy content, because customer records are architecturally absent from that collection. A filter dependency on a single mixed collection means a missing filter in any one code path exposes PII. Separate collections provide the guarantee by architecture, not by convention, which holds even when application code is wrong.

### Hidden Concepts Worth Knowing

- **Embedding vectors can leak information about the original text**: recent research has shown that under certain conditions, the original text that produced an embedding vector can be partially reconstructed from the vector itself. This is an emerging research area rather than a current widespread practical risk, but worth knowing when evaluating whether embedding PII carries additional risk beyond the plaintext payload.
- **Audit logging for vector search**: SQL databases have query logs. A Qdrant instance by default does not log which payloads were returned for which queries. For compliance-sensitive applications, application-layer audit logging (log every search, the filter applied, the caller identity, and the IDs of points returned) is necessary to reconstruct "who retrieved what and when" — which a direct Qdrant query log won't give you.

### Revision Summary

> PII in `page_content` means the vector store inherits the source database's security requirements — access control does not stop at the SQL boundary. The three risks specific to embedding PII: cross-customer leakage via semantic similarity, unintended surfacing via general queries against a mixed collection, and prompt injection via retrieved PII content. The two-layer fix: separate collection for customer records (architectural separation, not filter dependency), plus caller-scoped payload filtering within that collection. Payload filtering is application-layer logic, not infrastructure-layer enforcement — true access control requires both.

In [1]:
"""
qdrant_pii_scoping.py
-----------------------
PII and access-scoping in practice.

Shows:
  1. The risk: unscoped search leaks another customer's PII
  2. The wrong fix: doc_type filter in a mixed collection (still fragile)
  3. The right fix: separate collection + caller-scoped filtering
  4. Right-to-be-forgotten: deleting a point when a customer requests it
  5. Redaction before embedding: only embed non-PII fields for semantic
     search, keep PII in the structured database only

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    PayloadSchemaType,
    Filter,
    FieldCondition,
    MatchValue,
    PointIdsList,
)
from sentence_transformers import SentenceTransformer

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
VECTOR_SIZE = 384

# two collections -- the key architectural decision
POLICY_COLLECTION = "fd_policy_docs"      # public knowledge, no PII
CUSTOMER_COLLECTION = "fd_customer_records"  # PII, caller-scoped

client = QdrantClient(":memory:")
model = SentenceTransformer(MODEL_NAME)

# customer records -- what fd_master_database.csv rows look like as Documents
CUSTOMER_RECORDS = [
    {
        "fd_no": "BJ2019FD7717",
        "customer_name": "Shobha Chopra",
        "mobile": "9876543210",
        "account_no": "01845146270482",
        "amount_inr": 391000,
        "status": "Closed (Premature)",
        "branch": "Pune",
        "tenure_months": 24,
    },
    {
        "fd_no": "BJ2021FD4432",
        "customer_name": "Ramesh Patel",
        "mobile": "9823456781",
        "account_no": "02934857163920",
        "amount_inr": 250000,
        "status": "Active",
        "branch": "Pune",
        "tenure_months": 24,
    },
    {
        "fd_no": "BJ2022FD8891",
        "customer_name": "Anita Sharma",
        "mobile": "9765432109",
        "account_no": "03847261938471",
        "amount_inr": 500000,
        "status": "Active",
        "branch": "Mumbai",
        "tenure_months": 36,
    },
]

# policy chunks -- no PII
POLICY_CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "What is the penalty for early FD closure? A 1 percent penalty applies.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
]


def make_point_id(key: str) -> int:
    """Deterministic ID from any string key -- same input always gives same ID."""
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)


def record_to_text(record: dict) -> str:
    """Serialise a customer record to the same 'key: value' format the CSV
    loader produces -- this is what gets embedded."""
    return "\n".join(f"{k}: {v}" for k, v in record.items())


def record_to_redacted_text(record: dict) -> str:
    """Embed only non-PII fields -- branch, tenure, status, amount tier.
    PII (name, mobile, account_no) stays in the structured DB only."""
    amount_tier = "large" if record["amount_inr"] >= 300000 else "medium" \
        if record["amount_inr"] >= 100000 else "small"
    return (
        f"branch: {record['branch']}\n"
        f"tenure_months: {record['tenure_months']}\n"
        f"status: {record['status']}\n"
        f"amount_tier: {amount_tier}"
    )


def setup_policy_collection() -> None:
    """Policy docs -- public knowledge, no PII, no caller scoping needed."""
    client.create_collection(
        collection_name=POLICY_COLLECTION,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    texts = [c["text"] for c in POLICY_CHUNKS]
    embeddings = model.encode(texts, normalize_embeddings=True)
    points = [
        PointStruct(
            id=make_point_id(POLICY_CHUNKS[i]["source"] + str(i)),
            vector=embeddings[i].tolist(),
            payload={"text": POLICY_CHUNKS[i]["text"],
                     "source": POLICY_CHUNKS[i]["source"],
                     "doc_type": POLICY_CHUNKS[i]["doc_type"]},
        )
        for i in range(len(POLICY_CHUNKS))
    ]
    client.upsert(collection_name=POLICY_COLLECTION, points=points)
    print(f"Policy collection: {len(points)} points (no PII)")


def setup_customer_collection(use_redaction: bool = False) -> None:
    """Customer records -- separate collection, caller-scoped.
    use_redaction=True embeds only non-PII fields; PII is payload-only."""
    client.create_collection(
        collection_name=CUSTOMER_COLLECTION,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )

    # add payload index on fd_no for fast caller-scoped filtering
    client.create_payload_index(
        collection_name=CUSTOMER_COLLECTION,
        field_name="fd_no",
        field_schema=PayloadSchemaType.KEYWORD,
    )

    # choose what to embed -- full record or redacted version
    texts = [
        record_to_redacted_text(r) if use_redaction else record_to_text(r)
        for r in CUSTOMER_RECORDS
    ]
    embeddings = model.encode(texts, normalize_embeddings=True)

    points = [
        PointStruct(
            id=make_point_id(CUSTOMER_RECORDS[i]["fd_no"]),
            vector=embeddings[i].tolist(),
            payload={
                # PII stored in payload -- protected by caller-scoped filter
                "fd_no": CUSTOMER_RECORDS[i]["fd_no"],
                "customer_name": CUSTOMER_RECORDS[i]["customer_name"],
                "mobile": CUSTOMER_RECORDS[i]["mobile"],
                "account_no": CUSTOMER_RECORDS[i]["account_no"],
                "amount_inr": CUSTOMER_RECORDS[i]["amount_inr"],
                "status": CUSTOMER_RECORDS[i]["status"],
                "branch": CUSTOMER_RECORDS[i]["branch"],
                "tenure_months": CUSTOMER_RECORDS[i]["tenure_months"],
            },
        )
        for i in range(len(CUSTOMER_RECORDS))
    ]
    client.upsert(collection_name=CUSTOMER_COLLECTION, points=points)
    label = "redacted embedding" if use_redaction else "full record embedding"
    print(f"Customer collection: {len(points)} points ({label})")


def demo_risk_unscoped_search() -> None:
    """THE RISK: searching customer records with no caller-scoped filter.
    A query about Shobha Chopra's FD also returns Ramesh Patel's record
    because it is semantically similar (same branch, same tenure)."""
    print("\n--- RISK: Unscoped search returns other customers' PII ---")
    query = "FD status for customer in Pune with 24 month tenure"
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # no filter -- searches all customer records
    results = client.query_points(
        collection_name=CUSTOMER_COLLECTION,
        query=query_vector,
        limit=3,
        with_payload=True,
    ).points

    for r in results:
        # returns records for multiple customers -- cross-customer leakage
        print(f"  score={r.score:.4f} | fd_no={r.payload['fd_no']} | "
              f"name={r.payload['customer_name']} | mobile={r.payload['mobile']}")
    print("  All 3 customers' PII returned -- unintended cross-customer leakage")


def demo_caller_scoped_search(authenticated_fd_no: str) -> None:
    """THE FIX: filter by the authenticated caller's fd_no.
    Only the record belonging to this caller is ever a valid result."""
    print(f"\n--- FIX: Caller-scoped search (authenticated as {authenticated_fd_no}) ---")
    query = "what is my FD status"
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # filter by the authenticated caller's fd_no -- set at authentication time
    results = client.query_points(
        collection_name=CUSTOMER_COLLECTION,
        query=query_vector,
        query_filter=Filter(
            must=[FieldCondition(key="fd_no", match=MatchValue(value=authenticated_fd_no))]
        ),
        limit=3,
        with_payload=True,
    ).points

    if results:
        r = results[0]
        print(f"  Returned: fd_no={r.payload['fd_no']} | "
              f"name={r.payload['customer_name']} | status={r.payload['status']}")
        print(f"  Only this customer's record returned -- no cross-customer leakage")
    else:
        print(f"  No record found for {authenticated_fd_no}")


def demo_policy_search_isolation() -> None:
    """Separate collections prevent customer PII from appearing in policy searches.
    A policy search against POLICY_COLLECTION can never return customer records
    because they are not in that collection -- architectural guarantee."""
    print("\n--- SEPARATION: Policy search cannot return customer PII ---")
    query = "premature withdrawal penalty Pune"
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=POLICY_COLLECTION,  # searches ONLY policy docs
        query=query_vector,
        limit=3,
        with_payload=True,
    ).points

    print(f"  Policy search returned {len(results)} result(s):")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")
    print("  No customer PII in these results -- separate collection enforces this")


def demo_right_to_be_forgotten(fd_no: str) -> None:
    """GDPR right-to-be-forgotten: delete the point from Qdrant when a
    customer requests deletion. Deterministic ID means we know exactly
    which point to delete without scanning the collection."""
    print(f"\n--- RIGHT-TO-BE-FORGOTTEN: Delete {fd_no} from vector store ---")

    # confirm the point exists before deletion
    point_id = make_point_id(fd_no)
    before = client.retrieve(
        collection_name=CUSTOMER_COLLECTION,
        ids=[point_id],
        with_payload=True,
        with_vectors=False,
    )
    print(f"  Before: point exists = {len(before) > 0}")
    if before:
        print(f"  PII stored: name={before[0].payload['customer_name']}, "
              f"mobile={before[0].payload['mobile']}")

    # delete the point -- this must also happen when the source CSV row is deleted
    client.delete(
        collection_name=CUSTOMER_COLLECTION,
        points_selector=PointIdsList(points=[point_id]),
    )

    # confirm deletion
    after = client.retrieve(
        collection_name=CUSTOMER_COLLECTION,
        ids=[point_id],
        with_payload=True,
        with_vectors=False,
    )
    print(f"  After:  point exists = {len(after) > 0}")
    print(f"  Point deleted from Qdrant -- must also delete from source CSV/DB")
    total = client.get_collection(CUSTOMER_COLLECTION).points_count
    print(f"  Collection now has {total} points (was {total + 1})")


# ======================================================================
# Run all demos in order
# ======================================================================

print("Setting up collections...")
setup_policy_collection()
setup_customer_collection(use_redaction=False)

# 1. show the risk of unscoped search
demo_risk_unscoped_search()

# 2. show caller-scoped search (the fix)
demo_caller_scoped_search(authenticated_fd_no="BJ2019FD7717")

# 3. show that policy search cannot return customer PII (separate collections)
demo_policy_search_isolation()

# 4. right-to-be-forgotten
demo_right_to_be_forgotten(fd_no="BJ2021FD4432")

# 5. show that after deletion, a scoped search for that fd_no returns nothing
print("\n--- After deletion: caller-scoped search for deleted record ---")
demo_caller_scoped_search(authenticated_fd_no="BJ2021FD4432")


Setting up collections...
Policy collection: 3 points (no PII)
Customer collection: 3 points (full record embedding)

--- RISK: Unscoped search returns other customers' PII ---
  score=0.6793 | fd_no=BJ2021FD4432 | name=Ramesh Patel | mobile=9823456781
  score=0.6760 | fd_no=BJ2019FD7717 | name=Shobha Chopra | mobile=9876543210
  score=0.5544 | fd_no=BJ2022FD8891 | name=Anita Sharma | mobile=9765432109
  All 3 customers' PII returned -- unintended cross-customer leakage

--- FIX: Caller-scoped search (authenticated as BJ2019FD7717) ---
  Returned: fd_no=BJ2019FD7717 | name=Shobha Chopra | status=Closed (Premature)
  Only this customer's record returned -- no cross-customer leakage

--- SEPARATION: Policy search cannot return customer PII ---
  Policy search returned 3 result(s):
  score=0.6699 | [policy] Premature withdrawal incurs a 1 percent penalty on the appli
  score=0.5702 | [faq] What is the penalty for early FD closure? A 1 percent penalt
  score=0.2834 | [product] Senior citi

C:\Users\pauls\AppData\Local\Temp\ipykernel_22256\3759776254.py:142: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(
